# Week 6 - Final Deliverable: Semantic Search with ChromaDB

This notebook is the final working script for the internship deliverable.

It:
1. Creates a small document collection.
2. Creates embeddings.
3. Stores them in a local ChromaDB vector database.
4. Accepts sample questions.
5. Returns the most relevant chunks.


In [ ]:
# Install once if needed:
!pip install chromadb sentence-transformers


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load the embedding model.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create a local persistent vector database.
client = chromadb.PersistentClient(path="./week6_vector_db")

# Create/open our collection.
collection = client.get_or_create_collection(name="documents")

print("Database ready!")


In [ ]:
# Small document collection.
documents = [
    "Artificial Intelligence allows computers to perform tasks that normally require human intelligence.",
    "Machine learning allows a computer to learn patterns from examples and data.",
    "Natural language processing helps computers understand and generate human language.",
    "Vector databases store embeddings and support semantic similarity search.",
    "Semantic search retrieves information based on meaning rather than exact keywords.",
    "AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.",
    "Constraint satisfaction problems represent variables, possible values, and constraints.",
    "Genetic algorithms improve solutions through selection, crossover, and mutation."
]

# Give every document a simple unique ID.
ids = [f"chunk_{i}" for i in range(len(documents))]

# Create one embedding for every document.
embeddings = model.encode(documents, convert_to_numpy=True)

# Save everything into ChromaDB.
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

print("Stored chunks:", collection.count())


In [ ]:
def search_documents(question, top_k=3):
    # Convert the question into the same vector space as our documents.
    question_embedding = model.encode(question).tolist()

    # Ask ChromaDB for the closest vectors.
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    print("\nQuestion:", question)
    print("=" * 70)

    for i, document in enumerate(results["documents"][0], start=1):
        distance = results["distances"][0][i - 1]

        print(f"\nResult {i}")
        print(f"Distance: {distance:.4f}")
        print("Chunk:", document)

    return results


In [ ]:
# Sample queries required for the Week 6 task.

search_documents("How can AI create a timetable?")
search_documents("What is semantic search?")
search_documents("How does a genetic algorithm improve a solution?")



**Documents**
→ **Chunks**
→ **Embeddings**
→ **ChromaDB**
→ **Question embedding**
→ **Similarity search**
→ **Relevant chunks**

